## Conceitos básicos de algebra linear em Python

### 1) `ndarray`, `shape`, `dtype`, `axis`

* **`ndarray`** é o contêiner principal do NumPy: um bloco contínuo de memória tipado (todos os elementos têm o mesmo tipo, o **`dtype`**).
* **`shape`**: a forma do tensor. Ex.:

  * vetor coluna/linha não existe “de verdade”: um vetor é shape `(n,)`.
  * matriz é shape `(m, n)`.
  * lote (*batch*) de vetores pode ser shape `(B, n)`.
* **`axis`**: índice da dimensão. Para `(m, n, p)`,

  * `axis=0` percorre o tamanho `m`,
  * `axis=1` percorre `n`,
  * `axis=2` percorre `p`.

In [4]:
import numpy as np
a = np.array([1,2,3])         # shape (3,), dtype int32 (p.ex.)
A = np.array([[1,2],[3,4]])   # shape (2,2)

print(a.shape)     # (3,)
print(A.shape)     # (2,2)
print(A.dtype)     # e.g. dtype('int32')


(3,)
(2, 2)
int32


### 2) Indexação e *slicing* (incluindo `[..., None]`, `[:, None, :]`)

* **Básico**: `A[i, j]` pega escalar, `A[i, :]` pega a **linha i** completa, `A[:, j]` pega a **coluna j**.

* **Fatias**: `start:stop:step` como em Python.

* **`None` / `np.newaxis`**: insere um **eixo de tamanho 1**.

  * `x` shape `(M,)`; `x[:, None]` vira `(M, 1)`; `x[None, :]` vira `(1, M)`.
  * Serve para alinhar shapes e habilitar *broadcasting*.

* **Reticências `...`**: representa “todos os eixos restantes”.

  * Se `T` tem shape `(A,B,C,D)`, então `T[..., j]` ≡ `T[:, :, :, j]`.

* **Combinações comuns**:

  * `N[:, None, :]`: se `N` é `(S, M)`, vira `(S, 1, M)` (insere eixo entre `S` e `M`).
  * `proj[..., None]`: se `proj` é `(S, K)`, vira `(S, K, 1)`.

Essas inserções permitem operações como `(S,1,M) - (S,K,M) -> (S,K,M)` por *broadcasting*.

In [9]:
x = np.arange(6).reshape(3,2)   # shape (3,2)
print(x)
print(x.shape)
print(x[:, None, :].shape)             # (3,1,2)
print(x[None, :, :].shape)             # (1,3,2)
print(x[..., None].shape)              # (3,2,1)

[[0 1]
 [2 3]
 [4 5]]
(3, 2)
(3, 1, 2)
(1, 3, 2)
(3, 2, 1)


### 3) *Broadcasting* (como shapes “conversam”)

**Regra de ouro**: alinha-se pela **direita**; duas dimensões são compatíveis se:

* são **iguais**, ou
* uma delas é **1** (pode “esticar”), ou
* uma está **ausente** (pode ser introduzida com `None`).

Exemplos típicos:

* Somar vetor a cada linha da matriz:

  * `A` shape `(m, n)`, `b` shape `(n,)` → `A + b`  ✅ (b é “esticado” para `(m, n)`).
* Subtrair *batch* de vetores de *batch* de matrizes:

  * `X` `(S, K, M)`, `v` `(M,)` → `X - v`  ✅ vira `(S, K, M) - (1,1,M)` por *broadcasting*.

Falhas comuns:

* `A (m,n)` e `b (m,)` em `A + b` ❌ — alinhar à direita dá `(n)` vs `(m)`. Corrija mudando `b` para `(m,1)`: `b[:, None]`.


### 4) Álgebra linear no NumPy

#### 4.1 Produto matricial `@`

* `x @ y` é produto de tensores conforme regras de “`matmul` generalizado”.
* Casos:

  * `(m,n) @ (n,p) -> (m,p)`
  * `(n,) @ (n,) -> ()` (escalar = produto interno)
  * `(S,n) @ (n,) -> (S,)` (produto de cada linha por o vetor)

In [10]:
A = np.array([[1.,2.],[3.,4.]])  # (2,2)
x = np.array([10., 20.])         # (2,)
A @ x   # -> (2,) = [1*10+2*20, 3*10+4*20]

array([ 50., 110.])

#### 4.2 `np.linalg` (decomposições e normas)

* `np.linalg.norm(X, axis=k)` → norma L2 ao longo do eixo `k`.
* `np.linalg.solve(A, b)` resolve (A x = b).
* `np.linalg.eig`, `svd`, etc.

In [ ]:
A = np.array([[3.,1.],[1.,2.]])
b = np.array([9.,8.])
x = np.linalg.solve(A,b)   # solução do sistema

print(f'A={A}')
print(f'b={b}')
print(f'x={x}')

A=[[3. 1.]
 [1. 2.]]
b=[9. 8.]
x=[2. 3.]
[[3. 4.]
 [0. 2.]]
[[5.]
 [2.]]
[[0.6 0.8]
 [0.  1. ]]


In [14]:
V = np.array([[3.,4.],[0.,2.]])
print("V:", V)
row_norms = np.linalg.norm(V, axis=1, keepdims=True)  # (2,1)
print("row_norms:", row_norms)
U = V / (row_norms + 1e-32)  # normaliza linhas
print("U:", U)

V: [[3. 4.]
 [0. 2.]]
row_norms: [[5.]
 [2.]]
U: [[0.6 0.8]
 [0.  1. ]]


#### 4.3 `einsum` (forma “tensórica” do produto)

* Escreve-se a **contração** explicitamente por índices.
* Exemplos equivalentes:

  * `A @ x` ↔ `np.einsum('ij,j->i', A, x)`
  * Produto interno `a·b` ↔ `np.einsum('i,i->', a, b)`
  * Batelada de produtos: `N @ dirs.T` ↔ `np.einsum('sm,km->sk', N, dirs)`

`einsum` é ótimo para clareza tensórica e às vezes dá ganhos de performance/memória.

### 5) Empilhar/concatenar, reshape, transpor, performance

* **Empilhar**:

  * `np.stack([a,b], axis=0)` cria um novo eixo: se `a,b` são `(n,)`, `stack` vira `(2,n)`.
  * `np.vstack` empilha por linhas: `(n,)` vira `(2,n)`; matrizes `(m,n)` viram `(m1+m2, n)` se compatíveis.
  * `np.hstack` empilha por colunas, idem.

* **Concatenar**:

  * `np.concatenate([A,B], axis=k)` cola ao longo do eixo `k`.

* **`reshape`**:

  * Muda *shape* sem copiar (quando possível). `-1` deixa NumPy inferir o tamanho.
  * `a.reshape(m, n)` precisa que o número total de elementos **bata**.

* **Transpor**:

  * `A.T` para 2D (troca eixos).
  * `np.swapaxes(T, i, j)` para tensores N-D.
  * `np.transpose(T, (permutação_de_eixos))` geral.

* **Performance (visão)**:

  * NumPy é rápido quando você **vetoriza** (operações em blocos) e evita laços Python.
  * Muitas operações retornam **views** (sem copiar). Cópias custam.

### 6) Distância perpendicular

Queremos (d_\perp(\mathbf{n}, \hat{\mathbf{z}})) onde (\hat{\mathbf{z}}=\mathbf{z}/|\mathbf{z}|) e (\mathbf{n}) é um ponto normalizado.

Para um **lote** de pontos (N) `(S,M)` e **K** direções `dirs` `(K,M)`:

```py
dirs = ref_points / (np.linalg.norm(ref_points, axis=1, keepdims=True) + 1e-32)  # (K,M)
proj = N @ dirs.T                     # (S,K)   escalar <n_i, dir_k>
proj_vec = proj[..., None] * dirs[None, ...]   # (S,K,1) * (1,K,M) -> (S,K,M)
res = N[:, None, :] - proj_vec        # (S,1,M) - (S,K,M) -> (S,K,M)
d = np.linalg.norm(res, axis=2)       # (S,K)
```

Aqui:

* `[..., None]` e `[:, None, :]` entram para **alinhar eixos** e permitir o *broadcasting* que replica cada vetor `n_i` em K cópias e cada direção `dir_k` em S cópias.
